In [2]:
from pydub import AudioSegment

from google.colab import files
uploaded = files.upload()

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Saving Pikachu_audio.mpeg to Pikachu_audio.mpeg


In [3]:


# Load the original audio file (replace with your file path)
input_file = "/content/Pikachu_audio.mpeg"

# Step 1: Convert to WAV
raw = AudioSegment.from_file(input_file)
raw.export("original.wav", format="wav")

# Step 2: Load WAV file
wav_audio = AudioSegment.from_wav("original.wav")

# Step 3: Create delays
delays = [0, 1000, 2000]  # milliseconds (0s, 1s, 2s)

for i in delays:
    # Create silence for delay
    gap = AudioSegment.silent(duration=i)

    # Add delay before audio
    delayed_audio = gap + wav_audio

    # Step 4: Export new WAV file
    output_file = f"output_delay_{i//1000}s.wav"
    delayed_audio.export(output_file, format="wav")

    print(f"Saved: {output_file}")

Saved: output_delay_0s.wav
Saved: output_delay_1s.wav
Saved: output_delay_2s.wav


In [8]:
import numpy as np
from pydub import AudioSegment

def audio_to_array(file):
    audio = AudioSegment.from_wav(file)
    samples = np.array(audio.get_array_of_samples())
    return samples.astype(np.float32)
    #returning as float32 so that dot product doesn't give integer overflow



# Load signals
sig_0 = audio_to_array("output_delay_0s.wav")
sig_1 = audio_to_array("output_delay_1s.wav")  # reference
sig_2 = audio_to_array("output_delay_2s.wav")

# Make all signals same length (pad with zeros)
max_len = max(len(sig_0), len(sig_1), len(sig_2))

def equal_length(sig):
    return np.pad(sig, (0, max_len - len(sig)))

sig_0 = equal_length(sig_0)
sig_1 = equal_length(sig_1)
sig_2 = equal_length(sig_2)

#reference signals
ref_sig_1=sig_1
ref_sig_2= sig_0 + sig_2


# Normalize
def normalize(x):
    return x / np.linalg.norm(x)
    #this operation also becomes easier if we convert our sample to float32

sig_0 = normalize(sig_0)
sig_1 = normalize(sig_1)
sig_2 = normalize(sig_2)
ref_sig_2=normalize(ref_sig_2)




In [9]:
corr_0 = np.dot(sig_0, ref_sig_1)
corr_1 = np.dot(sig_1, ref_sig_1)
corr_2 = np.dot(sig_2, ref_sig_1)

print("Correlation with 1s reference:")
print(f"0s vs 1s: {corr_0}")
print(f"1s vs 1s: {corr_1}")
print(f"2s vs 1s: {corr_2}")

corr_0_new = np.dot(sig_0, ref_sig_1)
corr_1_new = np.dot(sig_1, ref_sig_1)
corr_2_new= np.dot(sig_2, ref_sig_1)

print("Correlation with 0s+2s reference:")
print(f"0s vs 0s+2s: {corr_0_new}")
print(f"1s vs 0s+2s: {corr_1_new}")
print(f"2s vs 0s+2s: {corr_2_new}")

Correlation with 1s reference:
0s vs 1s: -3443.961669921875
1s vs 1s: 6325432.5
2s vs 1s: 816.7027587890625
Correlation with 0s+2s reference:
0s vs 0s+2s: -3443.961669921875
1s vs 0s+2s: 6325432.5
2s vs 0s+2s: 816.7027587890625
